# Transformar la data de resultados
1. Leer la tabla bronze `results`
1. Mantener solo las colunas requeridas para analítica (Drop `url` column)
1. Estandarizar los nombres de la columnas a  snake_case y traducir a español (`constructorId` → `id_constructor`, `driverId` → `id_piloto`, `raceName` → `nombre_carrera`, `positionText` → `posicion_final_texto`)
1. Renombrar las columnas a  un nombre más significativo y traducir a español  (`date` → `fecha_carrera`, `grid` → `posicion_grilla`, `laps` → `vueltas_completadas`, `number` → `numero_de tarjeta`, `position` → `posicion_final` , `season` -> `temporada`,  `round` -> `ronda`)
1. Excluir registros donde `temporada`, `ronda`, `id_constrcutor` o `id_piloto` sean nulos (validación clave del negocio)
1. Remover registros duplicados
1. Transformar valores de la columna `nombre carrera` a Title Case
1. Escribir la data transformada a la tabla silver `resultados`


#### 


In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
bronze_table = f"{catalogo}.{esquema_bronze}.results"
silver_table = f"{catalogo}.{esquema_silver}.results"

In [0]:
from pyspark.sql import functions as F

#### 1 & 4 - Leer la tabla bronze `results`, mantener solo las columnas requeridas y estandarizar los nombres de la columnas

In [0]:
results_df = (
  spark.table(bronze_table)
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingesta_timestamp",
                "archivo_fuente")
       .withColumnsRenamed({
            "constructorId": "id_constructor",
            "round": "numero_carrera",
             "points": "puntos",
            "status" :"estado",
            "season": "temporada",
            "driverId": "id_piloto",
            "raceName": "nombre_carrera",
            "date": "fecha_carrera",
            "grid": "posicion_grilla",
            "laps": "vueltas_completadas",
            "number": "numero_tarjeta",
            "position": "posicion_final",
            "positionText": "posicion_final_texto"
        })
)

#### 5 & 6 Aplicar validaciones de calida de datos 
- Excluir registros donde `temporada`, `ronda`, `id_constrcutor` o `id_piloto` sean nulos (validación clave del negocio)
- Remover registros duplicados

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("temporada").isNotNull() &
            F.col("numero_carrera").isNotNull() &
            F.col("id_constructor").isNotNull() &
            F.col("id_piloto").isNotNull() 
        )
        .dropDuplicates(["temporada", "numero_carrera", "id_constructor", "id_piloto"])
)

In [0]:
display(results_df.count() - results_valid_df.count())

In [0]:
#results_valid_df
from pyspark.sql.functions import when, col
df_replace_case =( results_valid_df.withColumn("estado", F.regexp_replace(F.col("estado"), "Laps", "Vueltas"))
                                   .withColumn("estado", F.regexp_replace(F.col("estado"), "Lap", "Vuelta"))
                                   .withColumn( "estado",
                                                 when(col("estado") =="Oil leak",  "Fuga de Aceite")
                                                .when(col("estado") =="Engine",  "Motor")
                                                .when(col("estado") =="Transmission",  "Caja de Velocidades")
                                                .when(col("estado") =="Gearbox",  "Caja de Cambios")
                                                .when(col("estado") =="Magneto",  "Magneto")
                                                .when(col("estado") =="Finished",  "Finalizado")
                                                .when(col("estado") =="Accident",  "Accidente")
                                                .when(col("estado") =="Collision",  "Choque")
                                                .when(col("estado") =="Safety Car",  "Seguridad")
                                                .when(col("estado") =="Safety car",  "Seguridad")
                                                .when(col("estado") =="Collision with wall",  "Choque con pared")
                                                .when(col("estado") =="Collision with barrier",  "Choque con barrera")
                                                .when(col("estado") =="Collision with car",  "Choque con coche")
                                                .when(col("estado") =="Collision with other car",  "Choque con otro coche")
                                                .when(col("estado") =="Collision with other vehicle",  "Choque con otro vehículo")
                                                .when(col("estado") =="Not classified",  "No clasificado")
                                                .when(col("estado")=="Water pressure",  "Presión de agua")
                                                .when(col("estado")=="Disqualified",  "Descalificado")
                                                .when(col("estado")=="Electrical",  "Falla Eléctrica")
                                                .when(col("estado")=="Hydraulics",  "Falla Hidráulica")
                                                .when(col("estado")=="Brakes",  "Frenos")
                                                .when(col("estado")=="Suspension",  "Suspensión")
                                                .when(col("estado")=="Tyre",  "Neumático")
                                                .when(col("estado")=="Tyres",  "Neumáticos")
                                                .when(col("estado")=="Wheel",  "Rueda")
                                                .when(col("estado")=="Wheel rim",  "Llanta")
                                                .when(col("estado")=="Fuel pressure",  "Presión de Combustible")
                                                .when(col("estado")=="Fuel system",  "Sistema de Combustible")
                                                .when(col("estado")=="Fuel leak",  "Fuga de Combustible")
                                                .when(col("estado")=="Overheating",  "Sobrecalentamiento")
                                                .when(col("estado")=="Radiator",  "Radiador")
                                                .when(col("estado")=="Throttle",  "Acelerador")
                                                .when(col("estado")=="Clutch",  "Embrague")
                                                .when(col("estado")=="Driveshaft",  "Árbol de Transmisión")
                                                .when(col("estado")=="Differential",  "Diferencial")
                                                .when(col("estado")=="Steering",  "Dirección")
                                                .when(col("estado")=="Spun off",  "Salida de Pista")
                                                .when(col("estado")=="Spin",  "Trompo")
                                                .when(col("estado")=="Withdrew",  "Retirado")
                                                .when(col("estado")=="Retired",  "Retirado")
                                                .when(col("estado")=="Mechanical",  "Falla Mecánica")
                                                .when(col("estado")=="Power Unit",  "Unidad de Potencia")
                                                .when(col("estado")=="Power unit",  "Unidad de Potencia")
                                                .when(col("estado")=="ERS",  "Sistema de Recuperación de Energía")
                                                .when(col("estado")=="Battery",  "Batería")
                                                .when(col("estado")=="Exhaust",  "Escape")
                                                .when(col("estado")=="Vibrations",  "Vibraciones")
                                                .when(col("estado")=="Fire",  "Incendio")
                                                .when(col("estado")=="Driver unwell",  "Piloto Indispuesto")
                                                
                                                .otherwise(col("estado"))
                                               )
                                     .withColumn( "posicion_final_texto",
                                                  when(col("posicion_final_texto") =="R",  "Retirado")
                                                  .when(col("posicion_final_texto") =="D",  "Descalificado")
                                                  .otherwise(col("posicion_final_texto"))
                                                 )
                                   
                                   )
                                   
                                    

#### 7 - Transformar valores de la columna `nombre carrera` a Title Case

In [0]:
results_final_df = (
    df_replace_case
        .withColumn('nombre_carrera', F.initcap(F.col("nombre_carrera")))
)

#### 8 - Escribir la data transformada a la tabla silver `resultados`

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))